In [1]:
import pandas as pd
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from sklearn.model_selection import train_test_split

# Скачиваем нужные словари NLTK
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)

# Настраиваем инструменты для английского языка
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def clean_text(text):
    # 1. Приведение к нижнему регистру
    text = str(text).lower()
    # 2. Удаление пунктуации и цифр (оставляем только английские буквы)
    text = re.sub(r'[^a-z\s]', '', text)
    # 3. Токенизация и удаление стоп-слов
    tokens = [word for word in text.split() if word not in stop_words]
    # 4. Лемматизация
    tokens = [lemmatizer.lemmatize(word) for word in tokens]
    return ' '.join(tokens)

# Загружаем датасет со спамом (кодировка latin-1 обязательна для этого файла)
df = pd.read_csv('../data/spam.csv', encoding='latin-1')

# В датасете много пустых колонок (Unnamed: 2, 3, 4). Берем только класс и текст.
df = df[['v1', 'v2']]
df.columns = ['label', 'text']

# Кодируем целевую переменную: spam = 1, ham (не спам) = 0
df['label'] = df['label'].map({'spam': 1, 'ham': 0})

# Применяем функцию очистки
df['cleaned_text'] = df['text'].apply(clean_text)

# Разделяем на train и test
X_train, X_test, y_train, y_test = train_test_split(df['cleaned_text'], df['label'], test_size=0.2, random_state=42)

print("Данные готовы. Вот первые строки очищенного текста:")
display(df[['text', 'cleaned_text']].head())

Данные готовы. Вот первые строки очищенного текста:


,text,cleaned_text
0,"Go until jurong point, crazy.. Available only ...",go jurong point crazy available bugis n great ...
1,Ok lar... Joking wif u oni...,ok lar joking wif u oni
2,Free entry in 2 a wkly comp to win FA Cup fina...,free entry wkly comp win fa cup final tkts st ...
3,U dun say so early hor... U c already then say...,u dun say early hor u c already say
4,"Nah I don't think he goes to usf, he lives aro...",nah dont think go usf life around though


In [2]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report

# Создаем пайплайн
nlp_pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(min_df=2, max_df=0.9)), # Ограничиваем редкие и частые слова
    ('clf', LogisticRegression(max_iter=1000))
])

# Обучаем модель
nlp_pipeline.fit(X_train, y_train)

# Предсказания
y_pred = nlp_pipeline.predict(X_test)

# Оцениваем
print("=== Метрики классификации СМС (1 - Spam, 0 - Ham) ===")
print(classification_report(y_test, y_pred))

=== Метрики классификации СМС (1 - Spam, 0 - Ham) ===
              precision    recall  f1-score   support

           0       0.96      1.00      0.98       965
           1       0.96      0.71      0.82       150

    accuracy                           0.96      1115
   macro avg       0.96      0.85      0.90      1115
weighted avg       0.96      0.96      0.95      1115



In [3]:
# Собираем результаты в удобную таблицу
results = pd.DataFrame({
    'original_text': df.loc[X_test.index, 'text'], # Берем оригинальный текст до очистки
    'cleaned_text': X_test.values,
    'true_label': y_test.values,
    'predicted': y_pred
})

# Фильтруем ошибки
errors = results[results['true_label'] != results['predicted']]

print(f"Всего ошибок найдено: {len(errors)}")
print("=== 5 примеров для анализа ===")
pd.set_option('display.max_colwidth', None)
display(errors.head(5))

Всего ошибок найдено: 48
=== 5 примеров для анализа ===


,original_text,cleaned_text,true_label,predicted
1044,"We know someone who you know that fancies you. Call 09058097218 to find out who. POBox 6, LS15HB 150p",know someone know fancy call find pobox lshb p,1,0
683,Hi I'm sue. I am 20 years old and work as a lapdancer. I love sex. Text me live - I'm i my bedroom now. text SUE to 89555. By TextOperator G2 1DA 150ppmsg 18+,hi im sue year old work lapdancer love sex text live im bedroom text sue textoperator g da ppmsg,1,0
4733,Buy Space Invaders 4 a chance 2 win orig Arcade Game console. Press 0 for Games Arcade (std WAP charge) See o2.co.uk/games 4 Terms + settings. No purchase,buy space invader chance win orig arcade game console press game arcade std wap charge see ocoukgames term setting purchase,1,0
4700,I liked the new mobile,liked new mobile,0,1
3116,Now am free call me pa.,free call pa,0,1


### Анализ ошибок NLP-модели

При анализе ложных срабатываний (ошибок) модели классификации СМС были выдвинуты следующие гипотезы:

1. **Использование "спам-слов" в обычных переписках (False Positive):** Если обычный пользователь (ham) использует слова вроде "free", "call", "urgent" в шутку или бытовом контексте ("I got a free ticket to the movie, call me!"), TF-IDF дает высокий вес этим признакам, и модель ошибочно помечает СМС как спам.
2. **Короткие нетипичные спам-сообщения (False Negative):** Некоторые спамеры маскируют рассылку под сообщения от живых людей (например, просто "Hi, are you there?"). После очистки от стоп-слов такие тексты становятся практически пустыми, и алгоритм принимает их за безопасные (ham).
3. **Потеря пунктуации:** Спам часто пестрит обилием восклицательных знаков ("!!!") и символов валют ("$$$"). Наша функция нормализации жестко удалила все спецсимволы через регулярные выражения, лишив модель мощного визуального признака спама.
4. **Опечатки и сленг:** Спамеры намеренно делают опечатки ("fre3", "c@ll"), чтобы обойти фильтры. В нашем пайплайне параметр `min_df=2` отсекает такие уникальные слова как редкие, из-за чего модель не может на них опереться.
5. **Числа и номера телефонов:** В спаме почти всегда присутствуют номера телефонов или короткие коды для отправки СМС. Из-за удаления цифр (`re.sub(r'[^a-z\s]')`) модель лишилась важнейшего индикатора рекламной рассылки.